# 10 · Summary statistics and balance

Card 10. Reads `data/processed/sample.parquet` (built by `05_match.py`, with CARs from
`06_car.py` and controls from `07_controls.py`) and writes:

- `output/tables/summary.tex`: **Table 1** (sample by year) and **Table 2** (descriptive statistics by group)
- `output/tables/balance.tex`: **Table 3** (balance test: difference in means, t-statistics, standardized differences)

Main sample: the **1:1 match** (each treated announcement and its closest control). Controls are
matched with replacement, so a control used by several treated firms counts once per use.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC = Path(os.environ.get("MEGACAP_PROC", ROOT / "data" / "processed"))
TABLES = ROOT / "output" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

s = pd.read_parquet(PROC / "sample.parquet")
s["reaction_day"] = pd.to_datetime(s["reaction_day"])
main = s[s["match_rank"] <= 1].copy()                # treated + closest control
main["neg"] = (main["surprise"] < 0).astype(float)
yr = main[main["role"] == "treated"].set_index("match_id")["reaction_day"].dt.year
main["year"] = main["match_id"].map(yr)              # controls take their treated firm's year
t, c = main[main["role"] == "treated"], main[main["role"] == "control"]
print(f"Treated rows {len(t):,}, control rows {len(c):,} (1:1 with replacement)")

Treated rows 12,568, control rows 12,568 (1:1 with replacement)


## Table 1: sample by year

In [2]:
all3 = s[s["role"] == "control"].assign(year=lambda d: d["match_id"].map(yr))
tab1 = pd.DataFrame({
    "Events": t.groupby("year")["event_id"].nunique(),
    "Treated ann.": t.groupby("year").size(),
    "Treated firms": t.groupby("year")["permno"].nunique(),
    "Control ann. (1:1)": c.groupby("year")["accession"].nunique(),
    "Control firms (1:1)": c.groupby("year")["permno"].nunique(),
    "Control ann. (1:3)": all3.groupby("year")["accession"].nunique(),
})
tab1.loc["Total"] = [t["event_id"].nunique(), len(t), t["permno"].nunique(),
                     c["accession"].nunique(), c["permno"].nunique(), all3["accession"].nunique()]
tab1 = tab1.astype(int)
display(tab1)

,Events,Treated ann.,Treated firms,Control ann. (1:1),Control firms (1:1),Control ann. (1:3)
year,,,,,,
2018,19,1517,812,858,636,1489
2019,20,1543,800,978,705,1731
2020,15,1124,685,817,652,1684
2021,16,1481,798,1028,770,2096
2022,18,1664,779,1155,798,2256
2023,18,1862,932,1183,834,2274
2024,17,1735,1003,1185,917,2367
2025,16,1642,974,1106,846,2215
Total,139,12568,1781,8310,2060,16112


## Table 2: descriptive statistics

SUE, CARs and past return in percent. CARs are outcomes, shown for reference; the balance test in Table 3
focuses on pre-treatment variables.

In [3]:
VARS = [  # column, label, scale
    ("sue", r"SUE (\% of price)", 100),
    ("neg", "Negative surprise (share)", 1),
    ("logsize", "Log market cap (\\$M)", 1),
    ("bm", "Book-to-market", 1),
    ("log_analysts", "Log(1 + analysts)", 1),
    ("pastret", r"Past return, months $-12$ to $-2$ (\%)", 100),
    ("samedaycount", "Same-day announcements", 1),
    ("car01", r"CAR$(0,+1)$ (\%)", 100),
    ("car2_60", r"CAR$(+2,+60)$ (\%)", 100),
]
PRE = ["sue", "neg", "logsize", "bm", "log_analysts", "pastret", "samedaycount"]
OUTCOMES = ["car01", "car2_60"]

def describe(df):
    rows = {}
    for col, label, k in VARS:
        v = df[col].dropna() * k
        rows[col] = {"Mean": v.mean(), "Median": v.median(), "SD": v.std(), "N": len(v)}
    return pd.DataFrame(rows).T

tab2 = pd.concat({"Treated": describe(t), "Control": describe(c)}, axis=1)
display(tab2.round(3))

Treated                            Control                   \
                 Mean   Median      SD        N     Mean   Median      SD   
sue             0.124    0.080   0.884  12568.0    0.118    0.080   0.812   
neg             0.240    0.000   0.427  12568.0    0.233    0.000   0.423   
logsize         8.264    8.177   1.741  12566.0    8.289    8.204   1.714   
bm              0.497    0.396   0.410  11997.0    0.479    0.367   0.403   
log_analysts    2.140    2.197   0.688  12568.0    2.162    2.197   0.675   
pastret        17.350    8.920  48.818  12458.0   16.759    8.361  50.368   
samedaycount  170.265  170.000  65.329  12568.0  144.946  126.000  96.672   
car01           0.214    0.130   8.569  12495.0    0.341    0.359   8.956   
car2_60        -1.185   -0.981  17.598  12158.0   -1.656   -1.013  20.109   

                       
                    N  
sue           12568.0  
neg           12568.0  
logsize       12568.0  
bm            11862.0  
log_analysts  12568.0  
pastret       12397.0  
samedaycount  12568.0  
car01         12458.0  
car2_60       12124.0

## Table 3: balance test

Difference in means (treated − control) with Welch t-statistics, and the **standardized difference**
\((\bar x_T - \bar x_C)/\sqrt{(s_T^2 + s_C^2)/2}\). A common rule of thumb treats |std. diff| < 0.1 as
balanced. With about 12,000 firms per group, even tiny differences can have large t-statistics, so
the standardized difference is the better guide to economic size.

In [4]:
def balance(cols):
    out = {}
    for col, label, k in VARS:
        if col not in cols:
            continue
        a, b = t[col].dropna() * k, c[col].dropna() * k
        tt = stats.ttest_ind(a, b, equal_var=False)
        sd = (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)
        out[col] = {"Treated": a.mean(), "Control": b.mean(), "Diff": a.mean() - b.mean(),
                    "t-stat": tt.statistic, "p-value": tt.pvalue, "Std. diff": sd}
    return pd.DataFrame(out).T

bal_pre, bal_out = balance(PRE), balance(OUTCOMES)
display(bal_pre.round(3)); display(bal_out.round(3))
print(f"Share of negative surprises: treated {t['neg'].mean():.1%}, control {c['neg'].mean():.1%}")

,Treated,Control,Diff,t-stat,p-value,Std. diff
sue,0.124,0.118,0.006,0.516,0.606,0.007
neg,0.240,0.233,0.007,1.365,0.172,0.017
logsize,8.264,8.289,-0.025,-1.157,0.247,-0.015
bm,0.497,0.479,0.018,3.449,0.001,0.045
log_analysts,2.140,2.162,-0.022,-2.545,0.011,-0.032
pastret,17.350,16.759,0.590,0.938,0.348,0.012
samedaycount,170.265,144.946,25.319,24.328,0.000,0.307


,Treated,Control,Diff,t-stat,p-value,Std. diff
car01,0.214,0.341,-0.127,-1.141,0.254,-0.014
car2_60,-1.185,-1.656,0.471,1.941,0.052,0.025


Share of negative surprises: treated 24.0%, control 23.3%


## Write the LaTeX tables

In [5]:
LABEL = {col: label for col, label, _ in VARS}

def fmt(x, d=3):
    # number for LaTeX, with a proper minus sign
    if pd.isna(x):
        return ""
    s = f"{abs(x):,.{d}f}"
    return f"$-${s}" if x < 0 and float(s.replace(",", "")) != 0 else s

def stars(p):
    return "^{***}" if p < 0.01 else "^{**}" if p < 0.05 else "^{*}" if p < 0.1 else ""

# ---------- summary.tex: Table 1 + Table 2 ----------
L = [r"% Generated by notebooks/10_summary.ipynb. Requires \usepackage{booktabs}.",
     r"\begin{table}[htbp]", r"\centering", r"\small",
     r"\caption{Sample by year}", r"\label{tab:sample}",
     r"\begin{tabular}{lrrrrrr}", r"\toprule",
     r" & & \multicolumn{2}{c}{Treated} & \multicolumn{2}{c}{Control (1:1)} & Control (1:3) \\",
     r"\cmidrule(lr){3-4} \cmidrule(lr){5-6} \cmidrule(lr){7-7}",
     r"Year & Events & Ann. & Firms & Ann. & Firms & Ann. \\", r"\midrule"]
for y, r in tab1.iterrows():
    if y == "Total":
        L.append(r"\midrule")
    L.append(f"{y} & " + " & ".join(f"{v:,}" for v in r.values) + r" \\")
L += [r"\bottomrule", r"\end{tabular}", r"\par\medskip",
      r"\parbox{0.95\linewidth}{\footnotesize Events are mega-cap reaction days with at least one treated "
      r"announcement. Treated announcements have the same reaction day as Nvidia, Apple, Microsoft, Alphabet, "
      r"Amazon or Meta. Controls report 2 to 5 trading days from a mega-cap day, matched on FF12 industry, "
      r"market cap and SUE, with replacement. Ann.\ counts distinct announcements; firms are distinct PERMNOs. "
      r"The total for firms counts each firm once.}",
      r"\end{table}", "",
      r"\begin{table}[htbp]", r"\centering", r"\small", r"\setlength{\tabcolsep}{3.5pt}",
      r"\caption{Descriptive statistics, 1:1 matched sample}", r"\label{tab:descriptives}",
      r"\begin{tabular}{lrrrrrrrr}", r"\toprule",
      r" & \multicolumn{4}{c}{Treated} & \multicolumn{4}{c}{Control} \\",
      r"\cmidrule(lr){2-5} \cmidrule(lr){6-9}",
      r" & Mean & Median & SD & N & Mean & Median & SD & N \\", r"\midrule"]
for col in tab2.index:
    if col == "car01":
        L.append(r"\midrule \multicolumn{9}{l}{\emph{Outcomes}} \\")
    T, Cc = tab2.loc[col, "Treated"], tab2.loc[col, "Control"]
    L.append(f"{LABEL[col]} & {fmt(T['Mean'])} & {fmt(T['Median'])} & {fmt(T['SD'])} & {int(T['N']):,} & "
             f"{fmt(Cc['Mean'])} & {fmt(Cc['Median'])} & {fmt(Cc['SD'])} & {int(Cc['N']):,} \\\\")
L += [r"\bottomrule", r"\end{tabular}", r"\par\medskip",
      r"\parbox{0.95\linewidth}{\footnotesize SUE is actual EPS minus the median I/B/E/S consensus, divided by "
      r"the price before the reaction day, winsorized 1/99 by quarter. CARs use a market model estimated over "
      r"days $-250$ to $-30$. Book-to-market and past return are winsorized 1/99; book-to-market is missing "
      r"for negative book equity. Controls used for several treated firms are counted once per use.}",
      r"\end{table}"]
(TABLES / "summary.tex").write_text("\n".join(L) + "\n", encoding="utf-8")

# ---------- balance.tex: Table 3 ----------
B = [r"% Generated by notebooks/10_summary.ipynb. Requires \usepackage{booktabs}.",
     r"\begin{table}[htbp]", r"\centering", r"\small",
     r"\caption{Balance between treated and control announcements}", r"\label{tab:balance}",
     r"\begin{tabular}{lrrrrr}", r"\toprule",
     r" & Treated & Control & Difference & $t$-stat & Std.\ diff. \\", r"\midrule",
     r"\multicolumn{6}{l}{\emph{Panel A: Before the announcement}} \\"]
for tab in (bal_pre, bal_out):
    if tab is bal_out:
        B += [r"\midrule", r"\multicolumn{6}{l}{\emph{Panel B: Outcomes (for reference)}} \\"]
    for col, r in tab.iterrows():
        B.append(f"{LABEL[col]} & {fmt(r['Treated'])} & {fmt(r['Control'])} & "
                 f"${'-' if r['Diff'] < 0 else ''}{abs(r['Diff']):,.3f}{stars(r['p-value'])}$ & {fmt(r['t-stat'], 2)} & {fmt(r['Std. diff'])} \\\\")
B += [r"\bottomrule", r"\end{tabular}", r"\par\medskip",
      r"\parbox{0.95\linewidth}{\footnotesize Means for the 1:1 matched sample. Difference is treated minus "
      r"control, with Welch $t$-statistics; $^{*}$, $^{**}$, $^{***}$ denote significance at 10\%, 5\% and 1\%. "
      r"Std.\ diff.\ is the difference divided by $\sqrt{(s_T^2+s_C^2)/2}$; values below 0.1 in absolute "
      r"value are conventionally treated as balanced. Same-day announcements is higher for treated firms by "
      r"construction, since mega-cap days fall in peak earnings weeks, and enters the regressions as a control.}",
      r"\end{table}"]
(TABLES / "balance.tex").write_text("\n".join(B) + "\n", encoding="utf-8")
print("Wrote output/tables/summary.tex and output/tables/balance.tex")

Wrote output/tables/summary.tex and output/tables/balance.tex


## Is the sample balanced?

**Yes, the groups are balanced on everything a firm could choose or that matching targeted.** SUE is almost identical in the two groups (0.124% vs 0.118% of price, t = 0.52, standardized difference 0.007), and so is the share of negative surprises (24.0% vs 23.3%, t = 1.37), so there is no sign that firms pick mega-cap days to hide bad news; every other pre-announcement variable has a standardized difference below 0.05, and the two that are statistically significant (book-to-market and analyst coverage) are significant only because of the large sample, not because the gap is economically meaningful.

The one real difference is the same-day announcement count (170 vs 145, standardized difference 0.31), which is higher for treated firms by construction because mega-cap days fall in peak earnings weeks; it enters every regression as a control, together with its interaction with SUE.